# MMLU Trusted Reference Run (gate adjudication)

Adjudicates the active **STOP AND DEBUG** gate from the 2026-07-11 FP16 chat validation
(MMLU `172/400 = 0.430` vs the declared `0.55-0.65` range). Implements
`docs/MMLU_REFERENCE_RUN.md`: run `lm_eval==0.4.12` on the exact preserved model snapshot
(`Qwen/Qwen2.5-1.5B-Instruct` @ `989aa7980e4cf806f80c7fef2b1adb7bc71aa306`), same four
subjects, first 100 test items each, zero-shot, chat template — then compare all 400
predictions item-by-item against the archived pilot records, and run the five-shot
comparability anchor.

**This notebook does not rerun the pilot, change the registered protocol, or run any
quantization.** It produces evidence for a human-written gate decision record.

Setup: attach the `compression-eval-pilot-code` Dataset, select a **T4 GPU**, enable
**Internet**. Optional: expose a Kaggle secret named `HF_TOKEN` to avoid anonymous Hub
throttling. Expected runtime: roughly 20-40 minutes.

Interpretation was fixed before seeing the reference result (`docs/MMLU_REFERENCE_RUN.md`):

- Agreement >= 95% **and** reference accuracy near 0.43 → implementation validated;
  the absolute 0.55-0.65 expectation was not comparable; derive a new gate from this
  reference plus a documented tolerance.
- Agreement < 95% or materially higher reference accuracy → inspect the
  `pilot_b_disagreement` rows first (leading whitespace, length normalization,
  assistant-boundary span) before changing any code.
- B-label frequency alone is not evidence of a defect.

Run **every** cell regardless of outcome, save a version with outputs, and download
`/kaggle/working/kaggle_mmlu_reference_run.tar.gz`.

One reviewed deviation from the literal commands in `docs/MMLU_REFERENCE_RUN.md`: the
model is materialized once with `snapshot_download(..., max_workers=1)` into a fresh
`/kaggle/working/hf-cache` and `pretrained=` points at that local snapshot directory.
Hub-backed `from_pretrained` stalled indefinitely on this runtime in the completed
2026-07-11 run; the local-snapshot load is the validated recovery. The snapshot
directory name is asserted equal to the pinned revision, and both the Hub ID and
revision are recorded in the summary, so reference identity is unchanged.

In [ ]:
from pathlib import Path

roots = [p.parent for p in Path('/kaggle/input').glob('**/requirements.txt') if (p.parent / 'pilot_eval').is_dir()]
assert roots, 'Attach the repository Kaggle dataset before running this notebook.'
SOURCE = roots[0]
WORK = Path('/kaggle/working/compression-eval')
RESULTS = Path('/kaggle/working/results')
print('source:', SOURCE)

In [ ]:
import hashlib, json, shutil, tarfile
from collections import Counter

if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SOURCE, WORK)
RESULTS.mkdir(parents=True, exist_ok=True)

# Kaggle auto-extracts uploaded archives, so the attached dataset may hold the pilot
# records either as the original tarball or as an already-extracted directory.
# Content identity is verified by the preserved MMLU JSONL checksum either way.
EXPECTED_ARCHIVE_SHA = 'f22e4a6bcce2666a58fb6a4338889b969157db5edb20a122ef68243bfa08b739'
EXPECTED_MMLU_SHA = '99f8dd2493bc78d34dcada3029a95237af846da183112ec1e808cde982cde4c0'
ARCHIVE_NAME = 'kaggle_qwen25_1p5b_fp16_chat_validation.tar.gz'

candidates = sorted(SOURCE.glob('**/fp16.mmlu.jsonl'))
if candidates:
    PILOT_MMLU = candidates[0]
else:
    pilot_archive = SOURCE / ARCHIVE_NAME
    assert pilot_archive.is_file(), f'{ARCHIVE_NAME} (or its extracted contents) is missing from the attached dataset; rebuild and re-upload dist/kaggle_dataset.zip.'
    archive_sha = hashlib.sha256(pilot_archive.read_bytes()).hexdigest()
    assert archive_sha == EXPECTED_ARCHIVE_SHA, f'pilot archive checksum mismatch: {archive_sha}'
    extract_dir = RESULTS / 'kaggle_validation'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    with tarfile.open(pilot_archive) as tar:
        tar.extractall(extract_dir)
    PILOT_MMLU = extract_dir / 'kaggle_qwen25_1p5b_fp16_chat_validation' / 'fp16.mmlu.jsonl'
assert PILOT_MMLU.is_file(), f'missing {PILOT_MMLU}'
mmlu_sha = hashlib.sha256(PILOT_MMLU.read_bytes()).hexdigest()
assert mmlu_sha == EXPECTED_MMLU_SHA, f'pilot MMLU checksum mismatch: {mmlu_sha}'
print('pilot MMLU records:', PILOT_MMLU)

pilot_records = [json.loads(line) for line in PILOT_MMLU.read_text().splitlines() if line.strip()]
assert len(pilot_records) == 400, f'expected 400 pilot MMLU records, found {len(pilot_records)}'
pilot_correct = sum(r['correct'] for r in pilot_records)
assert pilot_correct == 172, f'expected 172 correct pilot records, found {pilot_correct}'
PILOT_ACCURACY = pilot_correct / len(pilot_records)
print(f'pilot MMLU: {pilot_correct}/400 = {PILOT_ACCURACY:.3f}')
print('pilot prediction counts:', dict(sorted(Counter(r['prediction'] for r in pilot_records).items())))
per_subject = Counter()
per_subject_n = Counter()
for r in pilot_records:
    subject = r['item_id'].split(':')[0]
    per_subject[subject] += int(r['correct'])
    per_subject_n[subject] += 1
PILOT_SUBJECT_ACCURACY = {s: per_subject[s] / per_subject_n[s] for s in sorted(per_subject_n)}
print('pilot per-subject accuracy:', {s: round(a, 3) for s, a in PILOT_SUBJECT_ACCURACY.items()})

In [ ]:
import importlib.metadata
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lm_eval==0.4.12'], check=True)

import torch

ENV_RECORD = {
    'lm_eval': importlib.metadata.version('lm_eval'),
    'torch': torch.__version__,
    'transformers': importlib.metadata.version('transformers'),
    'datasets': importlib.metadata.version('datasets'),
    'accelerate': importlib.metadata.version('accelerate'),
    'python': sys.version.split()[0],
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
assert ENV_RECORD['lm_eval'] == '0.4.12', f"expected lm_eval 0.4.12, got {ENV_RECORD['lm_eval']}"
assert ENV_RECORD['cuda_available'], 'select a GPU accelerator (T4) before running'
print(json.dumps(ENV_RECORD, indent=2))

In [ ]:
import os

os.environ['HF_HOME'] = '/kaggle/working/hf-cache'
os.environ['HF_HUB_DISABLE_XET'] = '1'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        print('HF_TOKEN secret attached')
except Exception:
    print('no HF_TOKEN secret; continuing anonymously')

from huggingface_hub import snapshot_download

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
REVISION = '989aa7980e4cf806f80c7fef2b1adb7bc71aa306'
SNAPSHOT = Path(snapshot_download(MODEL_ID, revision=REVISION, max_workers=1))
assert SNAPSHOT.name == REVISION, f'snapshot directory {SNAPSHOT.name} does not match pinned revision'
print('snapshot:', SNAPSHOT)

In [ ]:
import shlex, threading, time

RUN_ENV = {
    **os.environ,
    'HF_ENABLE_PARALLEL_LOADING': 'false',
    'CUDA_VISIBLE_DEVICES': '0',
    'TOKENIZERS_PARALLELISM': 'false',
}


def run_streaming(cmd, cwd=None, silence_timeout=600):
    """Run a command, stream its merged output, and kill it after silence_timeout seconds without output."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(shlex.quote(c) for c in cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, env=RUN_ENV, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    state = {'last': time.monotonic(), 'killed': False}

    def watchdog():
        while proc.poll() is None:
            if time.monotonic() - state['last'] > silence_timeout:
                state['killed'] = True
                proc.kill()
                return
            time.sleep(5)

    thread = threading.Thread(target=watchdog, daemon=True)
    thread.start()
    for line in proc.stdout:
        state['last'] = time.monotonic()
        print(line, end='', flush=True)
    proc.wait()
    thread.join(timeout=10)
    if state['killed']:
        raise RuntimeError(f'no output for {silence_timeout}s; process killed. Preserve outputs and rerun this cell.')
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(proc.returncode, cmd)
    return ' '.join(cmd)

In [ ]:
TASKS = 'mmlu_abstract_algebra,mmlu_college_computer_science,mmlu_high_school_statistics,mmlu_machine_learning'
OUT_0SHOT = RESULTS / 'lm_eval_mmlu_0shot'
COMMANDS = {}

COMMANDS['zero_shot'] = run_streaming([
    sys.executable, '-m', 'lm_eval',
    '--model', 'hf',
    '--model_args', f'pretrained={SNAPSHOT},dtype=float16',
    '--tasks', TASKS,
    '--num_fewshot', '0',
    '--limit', '100',
    '--batch_size', 'auto',
    '--apply_chat_template',
    '--log_samples',
    '--output_path', OUT_0SHOT,
], cwd=WORK)

In [ ]:
import csv, re

SAMPLES_NAME = re.compile(r'^samples_(?P<task>.+)_(?P<stamp>\d{4}-\d{2}-\d{2}T[\d\-.]+)\.jsonl$')


def normalize_samples(output_dir: Path, dest: Path) -> Path:
    """Copy harness samples files, injecting task_name from the filename when rows omit it."""
    files = sorted(Path(output_dir).rglob('samples_*.jsonl'))
    assert files, f'no harness samples files found under {output_dir}'
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True)
    for path in files:
        match = SAMPLES_NAME.match(path.name)
        assert match, f'unrecognized samples filename: {path.name}'
        rows = []
        for line in path.read_text().splitlines():
            if line.strip():
                row = json.loads(line)
                row.setdefault('task_name', match.group('task'))
                rows.append(row)
        (dest / path.name).write_text('\n'.join(json.dumps(r) for r in rows) + '\n')
    print(f'normalized {len(files)} samples files -> {dest}')
    return dest


def run_comparator(normalized_dir: Path, output_csv: Path) -> dict:
    proc = subprocess.run(
        [sys.executable, str(WORK / 'scripts/compare_mmlu_reference.py'),
         str(PILOT_MMLU), str(normalized_dir), '--output', str(output_csv)],
        capture_output=True, text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise subprocess.CalledProcessError(proc.returncode, proc.args)
    metrics = {}
    for line in proc.stdout.splitlines():
        if '=' in line:
            key, value = line.split('=', 1)
            metrics[key.strip()] = value.strip()
    return metrics


def subject_accuracy_from_diff(diff_csv: Path, column: str) -> dict:
    correct, totals = Counter(), Counter()
    with diff_csv.open() as handle:
        for row in csv.DictReader(handle):
            subject = row['item_id'].split(':')[0]
            totals[subject] += 1
            correct[subject] += row[column] == 'True'
    return {s: round(correct[s] / totals[s], 3) for s in sorted(totals)}


DIFF_0SHOT = RESULTS / 'mmlu_reference_diff_0shot.csv'
normalized_0shot = normalize_samples(OUT_0SHOT, RESULTS / 'lm_eval_mmlu_0shot_normalized')
METRICS_0SHOT = run_comparator(normalized_0shot, DIFF_0SHOT)
REFERENCE_SUBJECT_ACCURACY = subject_accuracy_from_diff(DIFF_0SHOT, 'reference_correct')
print('reference per-subject accuracy:', REFERENCE_SUBJECT_ACCURACY)
print('pilot     per-subject accuracy:', {s: round(a, 3) for s, a in PILOT_SUBJECT_ACCURACY.items()})

In [ ]:
OUT_5SHOT = RESULTS / 'lm_eval_mmlu_5shot'
FIVE_SHOT_ERROR = None
try:
    COMMANDS['five_shot'] = run_streaming([
        sys.executable, '-m', 'lm_eval',
        '--model', 'hf',
        '--model_args', f'pretrained={SNAPSHOT},dtype=float16',
        '--tasks', TASKS,
        '--num_fewshot', '5',
        '--limit', '100',
        '--batch_size', 'auto',
        '--apply_chat_template',
        '--log_samples',
        '--output_path', OUT_5SHOT,
    ], cwd=WORK)
except Exception as error:
    FIVE_SHOT_ERROR = f'{type(error).__name__}: {error}'
    print('FIVE-SHOT RUN FAILED (recorded; the zero-shot comparison above is the adjudicating evidence):')
    print(FIVE_SHOT_ERROR)


def harness_accuracies(output_dir: Path) -> dict:
    results_files = sorted(Path(output_dir).rglob('results_*.json'))
    if not results_files:
        return {}
    payload = json.loads(results_files[-1].read_text())
    accuracies = {}
    for task, values in payload.get('results', {}).items():
        for key in ('acc,none', 'acc'):
            if key in values:
                accuracies[task] = round(float(values[key]), 4)
                break
    return accuracies


FIVE_SHOT_ACCURACY = harness_accuracies(OUT_5SHOT) if FIVE_SHOT_ERROR is None else {}
ZERO_SHOT_HARNESS_ACCURACY = harness_accuracies(OUT_0SHOT)
print('zero-shot harness accuracies:', ZERO_SHOT_HARNESS_ACCURACY)
print('five-shot harness accuracies:', FIVE_SHOT_ACCURACY or FIVE_SHOT_ERROR)

In [ ]:
from datetime import datetime, timezone

reference_accuracy = float(METRICS_0SHOT['reference_accuracy'])
prediction_agreement = float(METRICS_0SHOT['prediction_agreement'])
pilot_b_reference_non_b = int(METRICS_0SHOT['pilot_B_reference_non_B'])

if prediction_agreement >= 0.95 and abs(reference_accuracy - PILOT_ACCURACY) <= 0.02:
    suggested_reading = (
        'IMPLEMENTATION VALIDATED: agreement >= 95% and reference accuracy is near 0.43. '
        'The 0.55-0.65 absolute expectation is not comparable to this zero-shot chat protocol; '
        'replace it with a gate derived from this reference plus a documented tolerance.'
    )
elif prediction_agreement < 0.95:
    suggested_reading = (
        'DISAGREEMENT: agreement < 95%. Inspect the pilot_b_disagreement rows in the diff CSV first '
        '(leading whitespace, continuation length normalization, assistant-boundary likelihood span) '
        'before changing any code.'
    )
else:
    suggested_reading = (
        'HIGH AGREEMENT BUT DIVERGENT ACCURACY: audit expected-range provenance and scoring '
        'differences; inspect the disagreement rows in the diff CSV.'
    )

summary = {
    'run_name': 'kaggle_mmlu_reference_run',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'purpose': 'Adjudicate the 2026-07-11 FP16 MMLU STOP AND DEBUG gate per docs/MMLU_REFERENCE_RUN.md.',
    'model_id': MODEL_ID,
    'revision': REVISION,
    'snapshot_path': str(SNAPSHOT),
    'pilot_archive_sha256': EXPECTED_ARCHIVE_SHA,
    'pilot_mmlu_sha256': EXPECTED_MMLU_SHA,
    'environment': ENV_RECORD,
    'commands': COMMANDS,
    'pilot_accuracy': PILOT_ACCURACY,
    'pilot_subject_accuracy': PILOT_SUBJECT_ACCURACY,
    'zero_shot': {
        'comparator_metrics': METRICS_0SHOT,
        'reference_accuracy': reference_accuracy,
        'prediction_agreement': prediction_agreement,
        'pilot_B_reference_non_B': pilot_b_reference_non_b,
        'reference_subject_accuracy': REFERENCE_SUBJECT_ACCURACY,
        'harness_reported_accuracy': ZERO_SHOT_HARNESS_ACCURACY,
    },
    'five_shot': {
        'harness_reported_accuracy': FIVE_SHOT_ACCURACY,
        'error': FIVE_SHOT_ERROR,
        'role': 'comparability anchor only; not a replacement for the registered zero-shot protocol',
    },
    'interpretation_rule': 'Fixed before execution in docs/MMLU_REFERENCE_RUN.md.',
    'suggested_reading': suggested_reading,
    'decision_record_written': False,
}
SUMMARY_PATH = RESULTS / 'reference_summary.json'
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

In [ ]:
ARCHIVE_OUT = RESULTS.parent / 'kaggle_mmlu_reference_run.tar.gz'
members = [OUT_0SHOT, RESULTS / 'lm_eval_mmlu_0shot_normalized', DIFF_0SHOT, SUMMARY_PATH]
if OUT_5SHOT.exists():
    members.append(OUT_5SHOT)
with tarfile.open(ARCHIVE_OUT, 'w:gz') as tar:
    for member in members:
        tar.add(member, arcname=str(member.relative_to(RESULTS.parent)))

checksums = {}
for path in (DIFF_0SHOT, SUMMARY_PATH, ARCHIVE_OUT):
    checksums[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
print(json.dumps(checksums, indent=2))
print(f'preserved archive: {ARCHIVE_OUT}')
print()
print(f'reference_accuracy={reference_accuracy:.4f}  prediction_agreement={prediction_agreement:.4f}  pilot_B_reference_non_B={pilot_b_reference_non_b}')
print(suggested_reading)
print()
print('NEXT: download the archive, save a notebook version with outputs, and write the gate decision')
print('record locally before any PACE bridge or main-grid work. This notebook does not resolve the gate by itself.')

## After this run

1. Download `/kaggle/working/kaggle_mmlu_reference_run.tar.gz` and save a Kaggle notebook
   version with outputs, whatever the result.
2. Back on the local machine, write the gate decision record (implementation fix vs
   corrected validation expectation, with the evidence) before touching PACE, the
   quantized bridge, or the main grid.
3. Independent of this gate, two implementation blockers remain before the main grid
   (see `STATUS.md` / `CODING_AGENT_HANDOFF_2026-07-10.md`): bring
   `scripts/build_quantized.py` into exact agreement with the registered 128 x 2,048-token
   calibration protocol, and implement the registered paired two-level seed-by-item
   bootstrap in `flipeval`.